In [9]:
import numpy as np
import requests
import re

# API Key untuk NewsAPI (Ganti dengan API key Anda)
API_KEY = "8711cc39b5ab4cf58b273b8fdbc6580b"
NEWS_API_URL = "https://newsapi.org/v2/everything?q=technology&language=en&apiKey=" + API_KEY

# Fungsi untuk mengambil berita dari NewsAPI
def get_news_articles():
    response = requests.get(NEWS_API_URL)
    if response.status_code == 200:
        data = response.json()
        articles = [article['title'] + ' ' + article['description'] for article in data['articles'] if article['description']]
        return ' '.join(articles)
    else:
        print("Gagal mengambil data dari NewsAPI")
        return ""

class SkipGramModel:
    def __init__(self, vocab_size, embedding_dim):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim

        # input dan hidden
        self.W1 = np.random.randn(vocab_size, embedding_dim)

        # hidden dan output
        self.W2 = np.random.randn(embedding_dim, vocab_size)

    def forward(self, one_hot_vector):
        hidden_layer = np.dot(one_hot_vector, self.W1)
        output_layer = np.dot(hidden_layer, self.W2)
        output_layer = self._softmax(output_layer)

        return hidden_layer, output_layer

    def backward(self, one_hot_vector, target_vector, learning_rate=0.01):
        hidden_layer, output_layer = self.forward(one_hot_vector)
        error = target_vector - output_layer

        # Compute Gradients
        output_layer_gradient = np.outer(hidden_layer, error)
        hidden_layer_gradient = np.outer(one_hot_vector, np.dot(self.W2, error))

        # Update Weights
        self.W1 -= learning_rate * hidden_layer_gradient
        self.W2 -= learning_rate * output_layer_gradient

    def _softmax(self, x):
        exp_x = np.exp(x-np.max(x))
        return exp_x / exp_x.sum()

# Fungsi untuk menyiapkan data dari NewsAPI
def prepare_training_data(window_size):
    text = get_news_articles()
    words = re.findall(r'\b\w+\b', text.lower())
    vocab = list(set(words))
    word2idx = {word: idx for idx, word in enumerate(vocab)}
    training_pairs = []

    for i, target_word in enumerate(words):
        target_idx = word2idx[target_word]
        for j in range(-window_size, window_size + 1):
            if j != 0 and 0 <= i + j < len(words):
                context_word = words[i + j]
                training_pairs.append((target_idx, word2idx[context_word]))

    return training_pairs, vocab, word2idx

def train_example(window_size=2, embedding_dim=100, epochs=10):
    training_pairs, vocab, word2idx = prepare_training_data(window_size)
    model = SkipGramModel(vocab_size=len(vocab), embedding_dim=embedding_dim)

    for epoch in range(epochs):
        total_loss = 0
        for target_idx, context_idx in training_pairs:
            target_vector = np.zeros(len(vocab))
            target_vector[target_idx] = 1

            context_vector = np.zeros(len(vocab))
            context_vector[context_idx] = 1

            hidden, output = model.forward(target_vector)
            loss = -np.log(output[context_idx])
            model.backward(target_vector, context_vector)
            total_loss += loss

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(training_pairs)}")

    return model, word2idx

if __name__ == "__main__":
    model, word2idx = train_example()

    # get word embeddings
    for word in word2idx:
        word_idx = word2idx[word]
        word_vector = model.W1[word_idx]
        print(f"{word}: {word_vector}")


C:\Users\Dell\AppData\Local\Temp\ipykernel_11284\1552808207.py:85: RuntimeWarning: divide by zero encountered in log
  loss = -np.log(output[context_idx])


Epoch 1/10, Loss: inf
Epoch 2/10, Loss: inf
Epoch 3/10, Loss: inf
Epoch 4/10, Loss: inf
Epoch 5/10, Loss: inf
Epoch 6/10, Loss: inf
Epoch 7/10, Loss: inf
Epoch 8/10, Loss: inf
Epoch 9/10, Loss: inf
Epoch 10/10, Loss: inf
too: [-1.22894313e+46 -3.96432704e+45 -2.16866371e+46 -3.98060961e+45
  7.24641500e+45 -3.96711542e+45  2.62737283e+45  4.79626386e+45
 -5.19280555e+45 -6.72859575e+45  5.56484964e+45 -1.79571032e+46
 -7.98379267e+45  2.05437800e+45  1.75001272e+46 -1.14962652e+46
 -7.68613051e+45 -2.09992256e+46  2.12611456e+44  1.50724196e+46
  1.67054309e+46  4.32474561e+45 -1.29828059e+46  2.24148241e+46
  1.48600431e+46 -2.17081425e+46 -2.67612780e+45  3.80922596e+45
 -1.54080550e+46  9.72204939e+45  4.50561979e+45 -4.16112597e+44
 -1.02029847e+45  1.20186472e+46  1.36923347e+46 -1.51870960e+46
  5.33391144e+45 -9.81028967e+45  1.13298768e+46 -2.89096007e+46
  1.14168823e+46 -6.27183346e+45  1.52688518e+46  3.26068142e+45
 -2.98132824e+45  1.00372564e+46  4.20720695e+45 -1.4077987